# Trust-region B: one-iteration diagnostic run

This notebook runs the isolated `stackelberg.trust_region_b` variant for exactly one trust-region iteration. It loads the final 120-dimensional NN price vector from `results/000`, enables one COPT network-state initialization before the local Adam solve, and writes all new outputs below `results/000/trust_region_b_one_iteration`.

In [1]:
import json
from pathlib import Path

import numpy as np
import pandas as pd
import pandapower.networks as pn

from stackelberg.followers.bg import create_cigre_residential_network
from stackelberg.followers.ip import create_industrial_park_network
from stackelberg.network import create_case33bw_20kv
from stackelberg.trust_region_b import (
    TrustRegionExperimentConfig,
    run_trust_region_experiment,
)

In [ ]:
T = 24
SCENARIOS = ["SB1", "SB2", "BG1", "BG2", "IP1"]
MG_BUSES = [17, 32, 15, 21, 1]
RESULTS_DIR = Path("results/000")
OUTPUT_DIR = RESULTS_DIR / "trust_region_b_one_iteration"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

history_candidates = sorted(RESULTS_DIR.glob("NN_solver_history_*.csv"))
if not history_candidates:
    raise FileNotFoundError(f"No NN history found below {RESULTS_DIR}.")
warm_start_path = max(history_candidates, key=lambda path: path.stat().st_mtime_ns)
nn_history = pd.read_csv(warm_start_path)
if nn_history.empty:
    raise ValueError(f"NN history is empty: {warm_start_path}")
pi_columns = sorted(
    (column for column in nn_history if column.startswith("pi_")),
    key=lambda column: int(column.removeprefix("pi_")),
)
if len(pi_columns) != len(SCENARIOS) * T:
    raise ValueError(
        f"Expected {len(SCENARIOS) * T} prices, got {len(pi_columns)}."
    )
last_nn_row = nn_history.sort_values("iteration").iloc[-1]
warm_start = last_nn_row[pi_columns].to_numpy(dtype=float)
nn_true_objective = float(last_nn_row["true_loss"])

print("Warm start:", warm_start_path)
print("Saved NN iteration:", int(last_nn_row["iteration"]))
print("Warm-start true objective:", nn_true_objective)
display(pd.DataFrame(warm_start.reshape(len(SCENARIOS), T), index=SCENARIOS))

In [ ]:
config = TrustRegionExperimentConfig(
    upper_net=create_case33bw_20kv(),
    lower_nets=[
        pn.create_cigre_network_lv(),
        pn.create_cigre_network_lv(),
        create_cigre_residential_network(),
        create_cigre_residential_network(),
        create_industrial_park_network(),
    ],
    mg_buses=MG_BUSES,
    scenarios=SCENARIOS,
    T=T,
    seed=42,
    device="cpu",
    x0=warm_start,
    lower_time_limit=30.0,
    lower_mip_gap=1e-3,
    copt_initialize_network_state=True,
    tr_params={
        "R_0": 0.05,
        "R_max": 1.0,
        "gamma_dec": 0.7,
        "gamma_inc": 1.5,
        "eta_1": 0.1,
        "eta_2": 0.75,
    },
    algo_params={
        "n_samples": 60,
        "K_max": 1,
        "sampling_method": "lhs",
        "residual_update_mode": "accumulate",
        "rbf_geometry_condition_max": 1e12,
        "random_seed": 42,
        "solve_params": {
            "steps_init": 0,
            "steps_warm": 2000,
            "lr": 1e-2,
            "penalty_weights": {
                "eq": 1000.0,
                "ineq": 100.0,
                "soc": 100.0,
                "bound": 10.0,
            },
            "tr_weight": 1e6,
            "verbose": False,
            "diagnostic_trace_every": 25,
        },
    },
    residual_model_params={
        "model_type": "rbf",
        "rbf_kernel": "cubic",
        "rbf_smoothing": 1e-3,
        "rbf_degree": 1,
        "rbf_scale": "trust_radius",
    },
    save_results=True,
    output_dir=str(OUTPUT_DIR),
    save_true_responses=False,
)

print("K_max:", config.algo_params["K_max"])
print("steps_warm:", config.algo_params["solve_params"]["steps_warm"])
print("COPT initialization:", config.copt_initialize_network_state)

In [ ]:
tr_result = run_trust_region_experiment(config)
print("Completed iterations:", tr_result["n_iter"])
print("Elapsed seconds:", tr_result["elapsed_seconds"])
print("Saved paths:", tr_result.get("saved_paths"))

In [ ]:
history = pd.DataFrame(tr_result["history"])
diagnostics = tr_result.get("optimizer_diagnostics") or {}
trace = pd.DataFrame(diagnostics.get("trace", []))
if not trace.empty:
    trace.to_csv(OUTPUT_DIR / "optimizer_trace.csv", index=False)

record = tr_result["history"][0]
summary = {
    "warm_start_path": str(warm_start_path.resolve()),
    "warm_start_true_objective": nn_true_objective,
    "F_current": float(record["F_current"]),
    "F_candidate": float(record["F_candidate"]),
    "F_tilde_current": float(record["F_tilde_current"]),
    "F_tilde_candidate": float(record["F_tilde_candidate"]),
    "rho": float(record["rho"]),
    "decision": record["decision"],
    "R_before": float(record["R_before"]),
    "R_after": float(record["R"]),
    "copt_warm_start_seconds": diagnostics.get("copt_warm_start_seconds"),
    "copt_warm_start_objective": diagnostics.get("copt_warm_start_objective"),
    "elapsed_seconds": float(tr_result["elapsed_seconds"]),
}
(OUTPUT_DIR / "one_iteration_summary.json").write_text(
    json.dumps(summary, indent=2), encoding="utf-8"
)
display(pd.DataFrame([summary]))
display(history)
display(trace)